<a href="https://colab.research.google.com/github/MellyMiranda/jarvis-academico-parte2/blob/main/Trabalho2_jarvis_academico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Jarvis Acadêmico**

In [1]:
# =========================
# IMPORTS E CONFIGURAÇÕES
# =========================

In [3]:
!apt-get install git
!pip install PyPDF2
!pip install sentence-transformers
!pip install scikit-learn
!pip install openai
!pip install gradio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [4]:
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import os
import json

In [5]:
import gradio as gr
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [6]:
!git clone https://github.com/MellyMiranda/jarvis-academico.git
pasta_pdfs = "jarvis-academico/data"

fatal: destination path 'jarvis-academico' already exists and is not an empty directory.


In [ ]:
##LLM ANTIGA
# client = OpenAI(base_url='https://llm.liaufms.org/v1/gemma-3-12b-it', api_key='Cxt2ftLF7d3mHS2JdiFqB-eSDAQeZvFATPXPs02lV9A')
# resp = client.chat.completions.create(
#     model='google/gemma-3-12b-it',
#     messages=[{'role': 'user', 'content': 'Hi'}],
# )
# print(resp.choices[0].message.content)


In [126]:
##LLM Nova
client = OpenAI(base_url='https://llm.liaufms.org/v1/qwen2-5-14b-instruct-awq', api_key='REIkURcI7rTTqsTwlJi8MrgnKFwOiqky7Ezh7hH-l-k')
resp = client.chat.completions.create(
    model='Qwen/Qwen2.5-14B-Instruct-AWQ',
    messages=[{'role': 'user', 'content': 'Olá'}],
)
print(resp.choices[0].message.content)


Olá! Como posso ajudar você hoje?


In [ ]:
# =========================
# RAG
# -carrega PDFs
# -tranforma em chunking
# -embeddings
# -resposta coerente
# =========================

In [8]:
documentos = []
for arquivo in os.listdir(pasta_pdfs): ##lendo pdfs

    if arquivo.endswith(".pdf"):

        caminho_completo = os.path.join(pasta_pdfs, arquivo)

        reader = PdfReader(caminho_completo)

        texto = ""

        for pagina in reader.pages:
            texto += pagina.extract_text()

        documentos.append({
            "arquivo": arquivo,
            "texto": texto
        })

        print(f"{arquivo} carregado com sucesso!")

4.Redes Neurais Artificiais.pdf carregado com sucesso!
10.Inteligência Artificial Generativa.pdf carregado com sucesso!
7.Deep Learning (DL).pdf carregado com sucesso!
1.Fundamentos de Machine Learning (1).pdf carregado com sucesso!
2.Introdução à Inteligência Artificial.pdf carregado com sucesso!
8.LLM.pdf carregado com sucesso!
9.Gradiente na Inteligência Artificial.pdf carregado com sucesso!
3.Regressão Logística.pdf carregado com sucesso!
5.RAG.pdf carregado com sucesso!
6.Embeddings.pdf carregado com sucesso!


In [9]:
chunks = []
tamanho_chunk = 500
overlap = 100
passo = tamanho_chunk - overlap

In [10]:
for doc in documentos:

    texto = doc["texto"]

    for i in range(0, len(texto), passo):

        chunk = texto[i:i + tamanho_chunk]

        chunks.append({
            "arquivo": doc["arquivo"],
            "texto": chunk
        })

In [11]:
modelo_embedding = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
for chunk in chunks:

    embedding = modelo_embedding.encode(chunk["texto"])

    chunk["embedding"] = embedding

In [13]:
def buscar_chunks(pergunta, top_k=3):

    embedding_pergunta = modelo_embedding.encode(pergunta)

    similaridades = []

    for chunk in chunks:

        similaridade = cosine_similarity(
            [embedding_pergunta],
            [chunk["embedding"]]
        )[0][0]

        similaridades.append({
            "arquivo": chunk["arquivo"],
            "texto": chunk["texto"],
            "similaridade": similaridade
        })

    similaridades = sorted(
        similaridades,
        key=lambda x: x["similaridade"],
        reverse=True
    )

    return similaridades[:top_k]

In [14]:
print(len(chunks))

134


In [ ]:
##Integracao Retrieval com Qwen

In [15]:
def montar_contexto(resultados):

    contexto = ""

    for resultado in resultados:

        contexto += resultado["texto"]
        contexto += "\n\n"

    return contexto

In [16]:
def responder_pergunta(pergunta):

    resultados = buscar_chunks(pergunta)

    contexto = montar_contexto(resultados)

    prompt = f"""
Você é um assistente acadêmico especializado em Inteligência Artificial e Machine Learning.

Responda a pergunta do usuário utilizando apenas o contexto fornecido.

Explique de maneira clara, organizada e didática.

Se a informação não estiver presente no contexto, diga que não encontrou informações suficientes.

Contexto:
{contexto}

Pergunta:
{pergunta}
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    return resposta.choices[0].message.content

In [17]:
def buscar_material_rag(pergunta):
    return responder_pergunta(pergunta)

In [ ]:
# ##TESTE-------------------------------
# resposta = responder_pergunta(
#     "Qual a diferença entre classificação e regressão?"
# )

# print(resposta)

In [ ]:
# ========================= MINIMO 5 OPCOES
# AGENDA ACADÊMICA: (implementadas)
# listar_agenda
# listar_tarefas
# adicionar_tarefa
# adicionar_evento(agenda)
# excluir tarefa
# excluir evento
# concluir_tarefa
# =========================

In [176]:
#TRABALHO1
agenda = []
tarefas = []
logs = []

# ACTIVE RECALL - TRABALHO2
topico_teste = ""
ultima_pergunta_teste = ""
em_teste = False
aguardando_resposta = False
perguntas_feitas = []

In [128]:
# =========================
# 1 - TAREFAS
# =========================

In [177]:
def salvar_tarefas():

    with open("jarvis-academico/tarefas.json", "w", encoding="utf-8") as arquivo:

        json.dump(
            tarefas,
            arquivo,
            ensure_ascii=False,
            indent=4
        )

In [178]:
def carregar_tarefas():

    global tarefas

    try:

        with open("jarvis-academico/tarefas.json", "r", encoding="utf-8") as arquivo:

            tarefas = json.load(arquivo)

        print("Tarefas carregadas com sucesso!")

    except:

        tarefas = []

        print("Nenhum arquivo de tarefas encontrado.")

In [179]:
carregar_tarefas()

Tarefas carregadas com sucesso!


In [180]:
def adicionar_tarefa(titulo, descricao):
    if not titulo:
        return "Erro: título não informado."

    tarefa = {
        "titulo": titulo,
        "descricao": descricao,
        "concluida": False
    }

    tarefas.append(tarefa)

    salvar_tarefas()

    return "Tarefa adicionada com sucesso!"

In [198]:
def listar_tarefas():

    if len(tarefas) == 0:
        return "Nenhuma tarefa cadastrada."

    texto = ""

    for i, tarefa in enumerate(tarefas):

        status = "✅ Concluída" if tarefa["concluida"] else "❌ Pendente"

        texto += f"""
ID: {i}
Título: {tarefa['titulo']}
Descrição: {tarefa['descricao']}
Status: {status}

"""

    return texto

In [199]:
def excluir_tarefa(id_tarefa):

    if 0 <= id_tarefa < len(tarefas):

        tarefas.pop(id_tarefa)
        salvar_tarefas()

        return "Tarefa excluída com sucesso!"

    return "ID inválido!"

In [200]:
def concluir_tarefa(id_tarefa):

    if 0 <= id_tarefa < len(tarefas):

        tarefas[id_tarefa]["concluida"] = True

        salvar_tarefas()

        return "Tarefa concluída com sucesso!"

    return "ID inválido!"

In [184]:
# =========================
# 2 - AGENDA
# =========================

In [201]:
def salvar_agenda():

    with open("jarvis-academico/agenda.json", "w", encoding="utf-8") as f:
        json.dump(agenda, f, ensure_ascii=False, indent=4)

In [202]:
def carregar_agenda():

    global agenda

    try:
        with open("jarvis-academico/agenda.json", "r", encoding="utf-8") as f:
            agenda = json.load(f)

        print("Agenda carregada com sucesso!")

    except:
        agenda = []
        print("Nenhum arquivo de agenda encontrado.")

In [203]:
carregar_agenda()

Agenda carregada com sucesso!


In [204]:
def listar_agenda(periodo="todos"):

  hoje = datetime.now(ZoneInfo("America/Campo_Grande"))

  eventos_filtrados = []

  for evento in agenda:

      # FORMATO: 20/05/2026
      data_evento = datetime.strptime(
          evento["data"],
          "%d/%m/%Y"
      )

      # HOJE
      if periodo == "hoje":

          if data_evento.date() == hoje.date():

              eventos_filtrados.append(evento)

      # AMANHÃ
      elif periodo == "amanha":

          amanha = hoje + timedelta(days=1)

          if data_evento.date() == amanha.date():

              eventos_filtrados.append(evento)

      # SEMANA
      elif periodo == "semana":

          diferenca = (data_evento.date() - hoje.date()).days

          if 0 <= diferenca <= 7:

              eventos_filtrados.append(evento)

      # TODOS
      else:

          eventos_filtrados.append(evento)

  if len(eventos_filtrados) == 0:

      return "Nenhum evento encontrado."

  texto = ""

  for evento in eventos_filtrados:

      texto += f"""

  ID: {evento['id']}
  Data: {evento['data']}
  Local: {evento['local']}
  Descrição: {evento['descricao']}

  """

  return texto


In [205]:
def adicionar_evento(data, local, descricao):

    if not data:
        return "Erro: data não informada."

    if not local:
        return "Erro: local não informado."

    evento = {
        "id": len(agenda),
        "data": data,
        "local": local,
        "descricao": descricao
    }

    agenda.append(evento)

    salvar_agenda()

    return "Evento adicionado com sucesso!"

In [206]:
def excluir_evento(id_evento):

    if 0 <= id_evento < len(agenda):

        agenda.pop(id_evento)
        salvar_agenda()

        return "Evento excluído com sucesso!"

    return "ID inválido!"

In [191]:
# =========================
# Funcionalidade 3.4 - TRABALHO2
# PLANEJAR ESTUDOS
# =========================

In [207]:
#Trabalho2:
tarefas_pendentes = []

for tarefa in tarefas:

    if not tarefa["concluida"]:
        tarefas_pendentes.append(tarefa)

In [208]:
# Trabalho 2
def planejar_estudos(pergunta):

    tarefas_pendentes = []

    # ------------------------
    # TAREFAS PENDENTES
    # ------------------------

    for tarefa in tarefas:

        if not tarefa["concluida"]:

            tarefas_pendentes.append(tarefa)

    # ------------------------
    # CONSULTA PARA O RAG
    # ------------------------

    consulta_rag = ""

    for tarefa in tarefas_pendentes:

        consulta_rag += tarefa["titulo"] + " "

    for evento in agenda:

        consulta_rag += evento["descricao"] + " "

    # Caso não existam tarefas nem eventos
    if consulta_rag.strip() == "":

        consulta_rag = pergunta

    # Busca materiais relacionados
    materiais = buscar_chunks(consulta_rag)

    contexto = ""

    # ------------------------
    # TAREFAS
    # ------------------------

    contexto += "TAREFAS PENDENTES\n"

    if len(tarefas_pendentes) == 0:

        contexto += "Nenhuma tarefa pendente.\n"

    else:

        for tarefa in tarefas_pendentes:

            contexto += (
                f"- {tarefa['titulo']} "
                f"({tarefa['descricao']})\n"
            )

    # ------------------------
    # AGENDA
    # ------------------------

    contexto += "\nAGENDA\n"

    hoje = datetime.now(
        ZoneInfo("America/Campo_Grande")
    )

    if len(agenda) == 0:

        contexto += "Nenhum evento cadastrado.\n"

    else:

        for evento in agenda:

            data_evento = datetime.strptime(evento["data"],
            "%d/%m/%Y").replace(tzinfo=ZoneInfo("America/Campo_Grande")
            )

            hoje_sem_hora = hoje.replace(
                hour=0,
                minute=0,
                second=0,
                microsecond=0
            )

            dias_restantes = (
                data_evento - hoje_sem_hora
            ).days

            contexto += (
                f"\nEvento: {evento['descricao']}\n"
                f"Data: {evento['data']}\n"
                f"Local: {evento['local']}\n"
                f"Dias restantes: {dias_restantes}\n"
            )

    # ------------------------
    # MATERIAIS RECUPERADOS
    # ------------------------

    contexto += "\nMATERIAIS RECUPERADOS\n"

    for material in materiais:

        contexto += material["texto"]
        contexto += "\n\n"

    return contexto

In [209]:
#trabalho2
def gerar_plano_estudos(pergunta, contexto):

    prompt = f"""
Você é um orientador acadêmico especializado em organização de estudos.

Seu objetivo é ajudar um estudante a decidir o que estudar primeiro.

Analise:

- tarefas pendentes
- eventos da agenda
- materiais recuperados pelo RAG

REGRAS:

1. Priorize provas e entregas mais próximas.
2. Considere tarefas ainda não concluídas.
3. Utilize os materiais recuperados como apoio.
4. Explique o motivo das prioridades.
5. Responda sempre em português.
6. Seja objetivo e organizado.

A resposta DEVE seguir exatamente este formato:

PRIORIDADE ALTA
- item 1
- item 2

PRIORIDADE MÉDIA
- item 1
- item 2

PLANO DE ESTUDO DE HOJE
- atividade 1
- atividade 2
- atividade 3

JUSTIFICATIVA
(explicação)

Dados:

{contexto}

Pedido do aluno:

{pergunta}
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.3
    )

    return resposta.choices[0].message.content

In [210]:
# =========================
# 5. MELHORIAS DE APRENDIZADO - TRABALHO2
# GERAR EXERCICIOS
# =========================

In [211]:
#TRABALHO2
def gerar_exercicios(pergunta):

    resultados = buscar_chunks(pergunta)

    contexto = montar_contexto(resultados)

    prompt = f"""
Você é um professor universitário.

Utilize o material fornecido para criar exercícios.

Regras:

- gere 5 questões
- misture questões fáceis e médias
- não forneça respostas
- responda em português

Material:

{contexto}

Tema pedido:

{pergunta}
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0.5
    )

    return resposta.choices[0].message.content

In [212]:
# =========================
# 5. MELHORIAS DE APRENDIZADO - TRABALHO2
# Active Recall (Interativa) TRABALHO2
# =========================

In [213]:
#TRABALHO2
def tool_iniciar_teste(pergunta):

    global em_teste, perguntas_feitas

    perguntas_feitas = []
    em_teste = True

    return iniciar_teste(pergunta)

In [214]:
#TRABALHO2
def iniciar_teste(pergunta):

    global modo_teste, topico_teste, ultima_pergunta_teste, perguntas_feitas

    resultados = buscar_chunks(pergunta)
    contexto = montar_contexto(resultados)

    prompt = f"""
Você é um professor.

Gere UMA pergunta DIFERENTE das anteriores.

Seja criativo e NÃO repita {perguntas_feitas} nem estrutura das perguntas anteriores.
Varie:
- contexto
- nível
- formulação

Material:
{contexto}
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8  # 🔥 importante aumentar criatividade
    )

    pergunta_gerada = resposta.choices[0].message.content

    # 🔥 GUARDA HISTÓRICO
    perguntas_feitas.append(pergunta_gerada)

    modo_teste = True
    topico_teste = pergunta
    ultima_pergunta_teste = pergunta_gerada

    return pergunta_gerada

In [238]:
#TRABALHO2
def avaliar_resposta(resposta_aluno):

    global ultima_pergunta_teste

    prompt = f"""
Você é um professor.

Pergunta feita:

{ultima_pergunta_teste}

Resposta do aluno:

{resposta_aluno}

Avalie:

- correta
- parcialmente correta
- incorreta

Explique brevemente.
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0
    )

    return (
      resposta.choices[0].message.content
      +"\n\n---\n\nDeseja continuar o teste?\nResponda: 'mais uma' ou 'encerrar'"
      )

In [216]:
#TRABALHO2
def interpretar_intencao_teste(texto):

    prompt = f"""
Você é um classificador de intenções.

O usuário está em um teste de aprendizado.

Classifique a mensagem em APENAS UMA categoria:

- encerrar
- continuar
- responder

REGRAS:
- "encerrar" = qualquer tentativa de parar o teste
- "continuar" = pedir nova pergunta (mais uma, próxima, etc)
- "responder" = qualquer resposta do aluno

Mensagem:
{texto}

Responda apenas com uma palavra.
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return resposta.choices[0].message.content.strip().lower()

In [217]:
# =========================
# LOGS
# =========================

In [218]:
def ver_logs():

    if len(logs) == 0:
        return "Nenhum log ainda."

    texto = ""

    for log in logs:

        texto += f"""
        Horário: {log['horario']}
        Tool: {log['tool']}
        Entrada:{log['entrada']}
        Saída:{log['saida']}
        Status: {log['status']}

        ==============================
        """

    return texto

In [219]:
def salvar_logs():

    with open("jarvis-academico/logs.json", "w", encoding="utf-8") as arquivo:

        json.dump(
            logs,
            arquivo,
            ensure_ascii=False,
            indent=4
        )

In [220]:
def carregar_logs():

    global logs

    try:

        with open("jarvis-academico/logs.json", "r", encoding="utf-8") as arquivo:

            logs = json.load(arquivo)

        print("Logs carregados com sucesso!")

    except:

        logs = []

        print("Nenhum arquivo de logs encontrado.")

In [221]:
carregar_logs()

Logs carregados com sucesso!


In [222]:
# =========================
# TOOL CALLING - IA decide qual função usar.
# =========================

In [223]:
tools = {
    #TRABALHO1
    #LISTAGENS
    "listar_tarefas": listar_tarefas,
    "listar_agenda": listar_agenda,
    #EXCLUSOES
    "excluir_tarefa": excluir_tarefa,
    "excluir_evento": excluir_evento,
    #ADICOES
    "adicionar_evento": adicionar_evento,
    "adicionar_tarefa": adicionar_tarefa,
    #CONCLUSOES
    "concluir_tarefa": concluir_tarefa,
    #BUSCA
    "buscar_material_rag": buscar_material_rag,

    #TRABALHO2
    "planejar_estudos": planejar_estudos,
    "gerar_exercicios": gerar_exercicios,
    "iniciar_teste": iniciar_teste,
    "avaliar_resposta": avaliar_resposta
}

In [225]:
def executar_tool(nome_tool, analise):

    # ERRO DE JSON / COMANDO
    if nome_tool == "erro":
        return analise["mensagem"]

    if nome_tool in tools:

        try:

            # RAG
            if nome_tool == "buscar_material_rag":
                resultado = tools[nome_tool](analise.get("pergunta", ""))

            # CONSULTA AGENDA
            elif nome_tool == "listar_agenda":
                resultado = tools[nome_tool](
                analise.get("periodo", "todos")
                )

            # TAREFA / AGENDA COM ID
            elif nome_tool in ["concluir_tarefa", "excluir_tarefa", "excluir_evento"]:
                id_item = analise.get("id") or analise.get("ID") or analise.get("numero") or 0
                resultado = tools[nome_tool](id_item)

            # ADICIONAR TAREFA
            elif nome_tool == "adicionar_tarefa":
                resultado = tools[nome_tool](
                    analise.get("titulo"),
                    analise.get("descricao")
                )

            # ADICIONAR EVENTO
            elif nome_tool == "adicionar_evento":
                resultado = tools[nome_tool](
                    analise.get("data"),
                    analise.get("local"),
                    analise.get("descricao")
                )

            #PLANEJAMENTO DE ESTUDOS
            elif nome_tool == "planejar_estudos":
                contexto = tools[nome_tool](
                    analise.get("pergunta", "")
                )

                resultado = gerar_plano_estudos(
                    analise.get("pergunta", ""),
                    contexto
                )

            #GERACAO DE EXERCICIOS
            elif nome_tool == "gerar_exercicios":
              resultado = tools[nome_tool](
              analise.get("pergunta", "")
                )

            #TESTE SOBRE ALGUM ASSUNTO
            elif nome_tool == "iniciar_teste":
              global em_teste, aguardando_resposta, topico_teste, perguntas_feitas

              em_teste = True
              aguardando_resposta = False
              topico_teste = analise.get("pergunta", "")
              perguntas_feitas = []

              resultado = iniciar_teste(topico_teste)

              aguardando_resposta = True  # entra no estado de resposta
              return resultado

            elif nome_tool == "avaliar_resposta":
              resultado = avaliar_resposta(
                  analise.get("resposta", "")
              )
              return resultado

            # LISTAGENS
            else:
                resultado = tools[nome_tool]()

            # LOG
            log = {
              "horario": str(datetime.now()),
              "tool": nome_tool,
              "entrada": analise,
              "saida": resultado,
              "status": "sucesso"
            }

            logs.append(log)
            salvar_logs()

            return resultado

        except Exception as e:
          log = {
              "horario": str(datetime.now()),
              "tool": nome_tool,
              "entrada": analise,
              "saida": str(e),
              "status": "erro"
          }

          logs.append(log)
          salvar_logs()

          return f"Erro ao executar tool: {e}"

    return "Tool não encontrada"

In [227]:
def analisar_pergunta(pergunta):

    prompt = f"""
Você é um sistema de roteamento de ferramentas.

Sua tarefa é escolher a ferramenta correta e extrair parâmetros quando necessário.

TOOLS DISPONÍVEIS:
- listar_tarefas
- listar_agenda
- buscar_material_rag
- concluir_tarefa
- excluir_tarefa
- excluir_evento
- adicionar_tarefa
- adicionar_evento

- planejar_estudos
- gerar_exercicios
- iniciar_teste
- avaliar_resposta

REGRAS:
- perguntas acadêmicas -> buscar_material_rag
- tarefas -> listar_tarefas
- agenda -> listar_agenda
- concluir tarefa -> concluir_tarefa
- excluir tarefa -> excluir_tarefa
- excluir evento -> excluir_evento
- adicionar tarefa -> adicionar_tarefa
- adicionar evento -> adicionar_evento

- planejamento de estudos -> planejar_estudos
- plano de estudos -> planejar_estudos
- priorizar estudos -> planejar_estudos
- exercicios -> gerar_exercicios
- questões -> gerar_exercicios
- gere perguntas -> gerar_exercicios
- me teste -> iniciar_teste
- faça uma pergunta como teste -> iniciar_teste
- active recall -> iniciar_teste

PERGUNTA:
{pergunta}
FORMATO DE RESPOSTA:
Responda SOMENTE em JSON válido.

EXEMPLOS:

Pergunta: Concluir tarefa 2
{{
  "tool": "concluir_tarefa",
  "id": 2
}}
Pergunta: Excluir evento 1
{{
  "tool": "excluir_evento",
  "id": 1
}}
Pergunta: Adicionar tarefa estudar IA, domingo
{{
  "tool": "adicionar_tarefa",
  "titulo": "estudar IA",
  "descricao":"domingo"
}}
Pergunta: Adicionar evento 20/05/2026, LAB 1, maratona prog
{{
  "tool": "adicionar_evento",
  "data": "20/05/2026",
  "local": "LAB 1",
  "descricao": "maratona prog"
}}
Pergunta: O que é regressão logística?
{{
  "tool": "buscar_material_rag",
  "pergunta": "O que é regressão logística?"
}}

Pergunta: O que tenho hoje?
{{
  "tool": "listar_agenda",
  "periodo": "hoje"
}}
Pergunta: Tenho prova amanhã?
{{
  "tool": "listar_agenda",
  "periodo": "amanha"
}}
Pergunta: Quais meus eventos da semana?
{{
  "tool": "listar_agenda",
  "periodo": "semana"
}}
Pergunta: Liste minha agenda
{{
  "tool": "listar_agenda",
  "periodo": "todos"
}}
Pergunta: Monte um plano de estudo para a prova
{{
  "tool": "planejar_estudos",
  "pergunta": "Monte um plano de estudo para a prova"
}}
Pergunta: O que devo priorizar hoje?
{{
  "tool": "planejar_estudos",
  "pergunta": "O que devo priorizar hoje?"
}}
Pergunta: O que estudar para a prova?
{{
  "tool": "planejar_estudos",
  "pergunta": "O que estudar para a prova?"
}}
Pergunta: Gere exercícios sobre regressão logística
{{
  "tool": "gerar_exercicios",
  "pergunta": "regressão logística"
}}
Pergunta: Faça questões sobre embeddings
{{
  "tool": "gerar_exercicios",
  "pergunta": "embeddings"
}}
Pergunta: Me teste sobre regressão logística
{{
  "tool": "iniciar_teste",
  "pergunta": "regressão logística"
}}

Agora responda:
Pergunta: {pergunta}
"""

    resposta = client.chat.completions.create(
        model='Qwen/Qwen2.5-14B-Instruct-AWQ',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0
    )

    texto = resposta.choices[0].message.content.strip()

    inicio = texto.find("{")
    fim = texto.rfind("}") + 1

    texto_json = texto[inicio:fim]

    try:
      return json.loads(texto_json)

    except Exception:
      return {"tool": "erro", "mensagem": "Não consegui entender o comando."
      }

In [228]:
def executar_comando(pergunta):

    analise = analisar_pergunta(pergunta)
    nome_tool = analise["tool"]

    return executar_tool(nome_tool, analise)

In [229]:
def jarvis(pergunta):

    global em_teste, aguardando_resposta

    if em_teste:

        intencao = interpretar_intencao_teste(pergunta)

        if intencao == "encerrar":
            em_teste = False
            aguardando_resposta = False
            return "🛑 Teste encerrado"

        if intencao == "continuar":
            aguardando_resposta = False
            return iniciar_teste(topico_teste)

        if aguardando_resposta:
            return avaliar_resposta(pergunta)

        aguardando_resposta = True
        return iniciar_teste(topico_teste)

    return executar_comando(pergunta)

In [263]:
# =========================
# TESTE GERAL TRABALHO 1
# =========================

# def teste_geral():

#     print("\n========================")
#     print("🧠 TESTE RAG")
#     print("PERGUNTA: O que é regressão logística?")
#     print("RESPOSTA:", jarvis("O que é regressão logística?"))

#     print("\n========================")
#     print("📋 TESTE ADICIONAR TAREFA")
#     print("COMANDO: Adicionar tarefa estudar KNN SABADO")
#     print("RESPOSTA: ", jarvis("Adicionar tarefa estudar IA DOMINGO"))

#     print("\n========================")
#     print("📋 TESTE LISTAR TAREFAS")
#     print("COMANDO: Listar minhas tarefas")
#     print("RESPOSTA: ", jarvis("Listar minhas tarefas"))

#     print("\n========================")
#     print("✅ TESTE CONCLUIR TAREFA")
#     print("COMANDO: Concluir tarefa 1")
#     print("RESPOSTA: ", jarvis("Concluir tarefa 1"))

#     print("\n========================")
#     print("🗑️ TESTE EXCLUIR TAREFA")
#     print("COMANDO: Excluir tarefa 0")
#     print("RESPOSTA: ",jarvis("Excluir tarefa 0"))

#     print("\n========================")
#     print("✅ TESTE MOSTRAR TAREFAS APOS EXCLUSAO")
#     print("COMANDO: Listar minhas tarefas")
#     print("RESPOSTA: ", jarvis("Listar minhas tarefas"))

#     print("\n========================")
#     print("📅 TESTE LISTAR AGENDA")
#     print("COMANDO: Listar minha agenda")
#     print("RESPOSTA: ",jarvis("Listar minha agenda"))

#     print("\n========================")
#     print("📅 TESTE ADICIONAR EVENTO")
#     print("COMANDO: Adicionar evento 20/06/2026, prova Estatistica, FACOM-SALA123")
#     print("RESPOSTA: ",jarvis("Adicionar evento 20/06/2026, prova Estatistica, FACOM-SALA123"))

#     print("\n========================")
#     print("📅 TESTE LISTAR AGENDA - APÓS ADIÇÃO")
#     print("COMANDO: agenda")
#     print("RESPOSTA: ",jarvis("agenda"))

#     print("\n========================")
#     print("🗑️ TESTE EXCLUIR EVENTO")
#     print("COMANDO: Excluir evento 0")
#     print("RESPOSTA: ", jarvis("Excluir evento 0"))

#     print("\n========================")
#     print("✅ TESTE MOSTRAR TAREFAS APOS EXCLUSAO")
#     print("COMANDO: agenda")
#     print("RESPOSTA: ",jarvis("agenda"))

#     print("\n========================")
#     print("\n========================")
#     print("📊 TESTE FINAL 1 COMPLETO")
#     print("========================")


# teste_geral()


🧠 TESTE RAG
PERGUNTA: O que é regressão logística?
RESPOSTA: A regressão logística é um método estatístico utilizado para modelar variáveis categóricas, especialmente aquelas que possuem apenas dois resultados possíveis (binárias). Ela é amplamente utilizada em diversos campos como análise de crédito, medicina, marketing, ciências sociais, previsão de risco e classificação de dados no contexto do machine learning.

Em termos mais simples, a regressão logística ajuda a prever a probabilidade de que uma observação pertença a uma categoria específica ou outra, baseando-se em um conjunto de variáveis independentes. Essa técnica é particularmente útil quando queremos prever um evento binário (como sim/não, aprovado/reprovado) com base em uma série de variáveis explicativas.

Ela é uma extensão da regressão linear, mas ao invés de prever valores contínuos, ela prevê a probabilidade de ocorrência de eventos categóricos.

📋 TESTE ADICIONAR TAREFA
COMANDO: Adicionar tarefa estudar KNN SABADO
R

In [264]:
# =========================
# TESTE GERAL TRABALHO 2
# =========================

# def teste_geral_parte2():

#     print("\n========================")
#     print("📚 TESTE GERAÇÃO DE EXERCÍCIOS (NOVO)")
#     print("========================")

#     print("COMANDO: gere exercícios sobre embeddings")
#     print("RESPOSTA:", jarvis("gere exercícios sobre embeddings"))

#     print("\nCOMANDO: faça questões sobre regressão logística")
#     print("RESPOSTA:", jarvis("faça questões sobre regressão logística"))

#     print("\n========================")
#     print("📅 TESTE PLANEJAMENTO DE ESTUDOS (3.4)")
#     print("========================")

#     print("COMANDO: monte um plano de estudos para prova de IA")
#     print("RESPOSTA:", jarvis("monte um plano de estudos para prova de IA"))

#     print("\nCOMANDO: o que devo priorizar hoje?")
#     print("RESPOSTA:", jarvis("o que devo priorizar hoje?"))

#     print("\n========================")
#     print("📋 TESTE TAREFAS + AGENDA (INTEGRAÇÃO)")
#     print("========================")

#     print("\n========================")
#     print("🧪 ACTIVE RECALL (INTERATIVO)")
#     print("========================")

#     print("COMANDO: me teste sobre regressão logística")
#     print("RESPOSTA:", jarvis("me teste sobre regressão logística"))

#     print("\nCOMANDO: não sei a resposta")
#     print("RESPOSTA:", jarvis("não sei a resposta"))

#     print("\nCOMANDO: mais uma")
#     print("RESPOSTA:", jarvis("mais uma"))

#     print("\nCOMANDO: eu nao sei responder")
#     print("RESPOSTA:", jarvis("eu nao sei responder"))

#     print("\nCOMANDO: encerra")
#     print("RESPOSTA:", jarvis("encerra"))

#     print("\n========================")
#     print("🧠 TESTES DE ROBUSTEZ ")
#     print("========================")

#     print("COMANDO: não sei estudar")
#     print("RESPOSTA:", jarvis("não sei estudar"))

#     print("\nCOMANDO: aaaaa")
#     print("RESPOSTA:", jarvis("aaaaa"))

#     print("\nCOMANDO: fazer isso")
#     print("RESPOSTA:", jarvis("fazer isso"))



#     print("\n========================")
#     print("\n========================")
#     print("📊 TESTE FINAL 2 COMPLETO")
#     print("========================")


# teste_geral_parte2()


📚 TESTE GERAÇÃO DE EXERCÍCIOS (NOVO)
COMANDO: gere exercícios sobre embeddings
RESPOSTA: 1. O que são embeddings em termos de Ciência de Dados e Inteligência Artificial?

2. Liste pelo menos duas vantagens dos embeddings na compreensão de dados semelhantes.

3. Explique como os embeddings são utilizados no processo de RAG (Retrieval Augmented Generation).

4. Como os embeddings ajudam computadores a entender relações e significados entre informações?

5. Descreva brevemente o processo de transformação de uma pergunta do usuário em um embedding e como esse embedding é usado para recuperar documentos relevantes.

COMANDO: faça questões sobre regressão logística
RESPOSTA: 1. Qual é a principal característica da regressão logística que a diferencia de outros métodos estatísticos?

2. Explique como a função logística é utilizada para modelar variáveis categóricas binárias.

3. No contexto da regressão logística, o que significa a razão entre a probabilidade de sucesso e a probabilidade de 

In [ ]:
# =========================
# 6. AVALIAÇÃO DO SISTEMA
# Ao menos 10 perguntas.
# Para cada pergunta:
# -pergunta
# -documentos recuperados
# -resposta
# -classificação:
#   **correta
#   **parcialmente correta
#   **incorreta
# =========================

In [267]:
# def avaliar_sistema():

#     avaliacoes = []

#     perguntas = [
#         "O que é regressão logística?",
#         "explique regressão logística de forma simples",
#         "gere exercícios sobre embeddings",
#         "faça questões sobre regressão logística",
#         "monte um plano de estudos para prova de IA",
#         "o que devo priorizar hoje?",
#         "adicionar tarefa estudar IA amanhã",
#         "listar minhas tarefas",
#         "o que tenho amanhã?",
#         "me teste sobre regressão logística",
#         "não sei responder",
#         "encerrar"
#     ]

#     for i, p in enumerate(perguntas):

#         print("\n========================")
#         print(f"PERGUNTA {i+1}: {p}")

#         resposta = jarvis(p)

#         print("RESPOSTA:")
#         print(resposta)

#         # avaliação manual (você ajusta depois)
#         print("\nCLASSIFICAÇÃO (preencher depois): correta / parcial / incorreta")

#         avaliacoes.append({
#             "pergunta": p,
#             "resposta": resposta,
#             "classificacao": "pendente"
#         })

#     return avaliacoes


# resultado_avaliacao = avaliar_sistema()


PERGUNTA 1: O que é regressão logística?
RESPOSTA:
A resposta do aluno está **incorreta**, pois ele respondeu a uma pergunta diferente da que foi feita. A pergunta era sobre o conceito de logit, mas o aluno respondeu sobre regressão logística.

Para explicar o conceito de logit em termos simples:

O logit é uma função matemática usada principalmente em modelos de regressão logística. Em essência, ela transforma probabilidades (que variam entre 0 e 1) em valores que podem variar de -∞ a +∞. Isso permite que os dados sejam modelados de maneira mais fácil e intuitiva. No contexto de uma análise estatística, o logit é usado para prever a probabilidade de um evento ocorrer baseado em uma ou mais variáveis independentes.

Agora, se quiséssemos responder à pergunta do aluno sobre regressão logística, poderíamos dizer algo como: "Regressão logística é um tipo de modelo estatístico usado quando queremos prever uma variável dependente categórica (geralmente binária, como sim/não ou 0/1) com bas

In [266]:
def avaliar_sistema():

    avaliacoes = []

    perguntas = [
        "O que é regressão logística?",
        "explique regressão logística de forma simples",
        "gere exercícios sobre embeddings",
        "faça questões sobre regressão logística",
        "monte um plano de estudos para prova de IA",
        "o que devo priorizar hoje?",
        "adicionar tarefa estudar IA amanhã",
        "listar minhas tarefas",
        "o que tenho amanhã?",
        "me teste sobre regressão logística"
    ]

    for i, p in enumerate(perguntas):

        print("\n========================")
        print(f"PERGUNTA {i+1}: {p}")

        resposta = jarvis(p)

        print("RESPOSTA:")
        print(resposta)

        # avaliação manual (ajustada depois no RELATÓRIO)
        print("\nCLASSIFICAÇÃO (preencher depois): correta / parcial / incorreta")

        avaliacoes.append({
            "pergunta": p,
            "resposta": resposta,
            "classificacao": "pendente"
        })

    return avaliacoes


resultado_avaliacao = avaliar_sistema()


PERGUNTA 1: O que é regressão logística?
RESPOSTA:
A regressão logística é um método estatístico utilizado para modelar variáveis categóricas, especialmente aquelas que possuem apenas dois resultados possíveis (binárias). Ela é frequentemente usada quando queremos prever uma probabilidade ou classificar dados em duas categorias.

Algumas características importantes da regressão logística incluem:

- **Aplicações**: A regressão logística é amplamente utilizada em várias áreas como análise de crédito, medicina, marketing, ciências sociais, previsão de risco, classificação de dados e machine learning.

- **Modelagem Binária**: Este modelo estatístico é uma extensão da regressão linear, mas é projetado especificamente para variáveis dependentes que são binárias (ex: sim/não, aprovado/reprovado).

- **Interpretação dos Coeficientes**: Os coeficientes da regressão logística indicam como a probabilidade de um evento específico ocorrer muda com relação às mudanças nas variáveis independentes.

In [232]:
# =========================
# INTERFACE (GRADIO)
# =========================

In [254]:
def chat_interface(mensagem, historico):

    resposta = jarvis(mensagem)

    historico.append({
        "role": "user",
        "content": mensagem
    })

    historico.append({
        "role": "assistant",
        "content": resposta
    })

    return historico, historico, ""

In [234]:
def limpar_chat():
    return []

In [257]:
def mensagem_inicial():

    global em_teste, topico_teste, perguntas_feitas
    em_teste = False
    topico_teste = ""
    perguntas_feitas = []

    return [
        {
            "role": "assistant",
            "content": """🤖 JARVIS ACADÊMICO

📌 MENU E COMO USAR (IMPORTANTE):

🟢 TAREFAS
Adicionar Tarefa:
→ Modelo: titulo, descricao

Listar Tarefas:
→ Modelo: Listar minhas tarefas

Concluir uma Tarefa:
→ Modelo: Concluir tarefa 0 (USE ID)

Excluir Tarefa:
→ Modelo: Excluir tarefa 0 (USE ID)

────────────────────

🔵 AGENDA
Adicionar Evento:
→ Modelo: data, local, descricao

Listar Agenda:
→ Modelo: Listar agenda

Eventos Hoje:
→ Modelo: O que tenho hoje?

Eventos Amanhã:
→ Modelo: Tenho algo amanhã?

Eventos da Semana:
→ Modelo: Quais meus eventos da semana?

Excluir Evento:
→ Modelo: Excluir evento 0 (USE ID)

────────────────────

📚 RAG (MATERIAIS)
→ O que é regressão logística?
→ Explique embeddings
→ Resuma redes neurais

────────────────────
"""
        }
    ]

In [260]:
css = """
body {
    font-size: 14px;
}

.gradio-container {
    max-width: 1400px !important;
}

.message {
    font-size: 14px !important;
}

textarea {
    font-size: 14px !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=css
) as demo:

    gr.Markdown("# 🤖 JARVIS ACADÊMICO")

    # =========================
    # CHAT
    # =========================

    with gr.Tab("💬 Chat"):

        chatbot = gr.Chatbot(
            type="messages",
            height=500,
            bubble_full_width=False,
            value=mensagem_inicial()
        )

        msg = gr.Textbox(
            label="Digite sua pergunta",
            placeholder="Ex: Adicionar tarefa, estudar IA, revisar embeddings",
            lines=1
        )

        state = gr.State([])
        em_teste_state = gr.State(False)
        topico_state = gr.State("")


        with gr.Row():
          send = gr.Button("Enviar",variant="primary")
        menu_btn = gr.Button("MENU/DÚVIDAS")
        clear = gr.Button("Limpar chat")

        send.click(
            chat_interface,
            [msg, state, em_teste_state, topico_state],
            [chatbot, state, msg, em_teste_state, topico_state]
        )

        msg.submit(chat_interface,[msg, state],[chatbot, state, msg])

        clear.click(limpar_chat,outputs=[chatbot])
        menu_btn.click(mensagem_inicial,outputs=[chatbot])

    # =========================
    # TAREFAS
    # =========================

    with gr.Tab("📋 Tarefas"):

        tarefas_btn = gr.Button("Atualizar tarefas")

        tarefas_out = gr.Textbox(lines=15)

        tarefas_btn.click(
            lambda: listar_tarefas(),
            outputs=[tarefas_out]
        )

    # =========================
    # AGENDA
    # =========================

    with gr.Tab("📅 Agenda"):

        agenda_btn = gr.Button("Atualizar agenda")

        agenda_out = gr.Textbox(lines=15)

        agenda_btn.click(
            lambda: listar_agenda(),
            outputs=[agenda_out]
        )

    # =========================
    # LOGS
    # =========================

    with gr.Tab("📊 Logs"):

        logs_btn = gr.Button("Ver logs")

        logs_out = gr.Textbox(lines=20)

        logs_btn.click(
            lambda: ver_logs(),
            outputs=[logs_out]
        )


/tmp/ipykernel_68241/1902326041.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_68241/1902326041.py:19: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_68241/1902326041.py:32: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_68241/1902326041.py:32: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(
/usr/local/lib/python3.12/dist-packages/gradio/utils.py:1052: UserWarning: Expected 2 arguments for function <fu

In [265]:
# =========================
# EXECUÇÃO
# =========================
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a369e092f183e7e1b6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [174]:
jarvis("me teste sobre regressão logística")

'Em um estudo sobre mobilidade ocupacional, você está usando regressão logística para prever se um indivíduo migrará de uma classe social para outra. Qual seria o significado prático de uma estimativa β (beta) positiva associada à variável independente "anos de educação"?'

In [175]:
avaliar_resposta(
    "A função logística converte o resultado das variáveis explicativas em uma probabilidade entre 0 e 1."
)

'A resposta do aluno é **parcialmente correta**.\n\nA explicação do aluno é parcialmente correta porque ela descreve corretamente a função logística, mas não responde diretamente à pergunta sobre o significado prático de uma estimativa β positiva associada à variável independente "anos de educação".\n\nUma resposta completa deveria incluir a explicação de que uma estimativa β positiva indica que, conforme aumentam os anos de educação, a probabilidade de um indivíduo migrar para uma classe social superior também aumenta. Isso implica que mais educação está associada a maiores chances de mobilidade social ascendente.\n\nPortanto, a resposta do aluno, embora contenha informações relevantes sobre a função logística, não aborda diretamente o significado prático da estimativa β positiva na questão proposta.'